# 🐾 KXD — Generador Furry & YiFF (Versión Infallible Multi-LoRA)

Instrucciones:
1. Asegúrate de activar la GPU: **Runtime › Change runtime type › T4 GPU**.
2. Ejecuta la **Celda 1** (Descarga, compila el mega-pack en CPU y carga el modelo).
3. Ejecuta la **Celda 2** para conectar con la web y procesar trabajos.

## ▶ Celda 1 — Instalación, Compilación del Pack y Carga

In [ ]:
import subprocess, sys, os, requests, torch, gc
from safetensors.torch import load_file, save_file

CIVITAI_API_KEY = "3a0a1d671ce384e4a599a84c51b8d250"

print('📦 Instalando dependencias de inferencia...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', 'torchao'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'diffusers>=0.27.0', 'transformers', 'accelerate', 'safetensors', 'torch', 'torchvision', 'xformers', 'fastapi', 'uvicorn[standard]', 'requests', 'Pillow'], check=True)

if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-q', '-O', '/tmp/cloudflared.deb', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], check=True)
    subprocess.run(['dpkg', '-i', '/tmp/cloudflared.deb'], check=True)
print('✅ Dependencias listas')

os.makedirs('/content/models', exist_ok=True)
os.makedirs('/content/loras',  exist_ok=True)

MODEL_PATH = '/content/models/sdxl_model.safetensors'
LORA_REGISTRY = {
    'zonkpunch_base': '/content/loras/zp_base.safetensors',
    'zonkpunch_1':    '/content/loras/zp_1.safetensors',
    'zonkpunch_2':    '/content/loras/zp_2.safetensors',
    'zonkpunch_3':    '/content/loras/zp_3.safetensors',
    'zonkpunch_4':    '/content/loras/zp_4.safetensors',
    'zonkpunch_5':    '/content/loras/zp_5.safetensors',
    'zonkpunch_6':    '/content/loras/zp_6.safetensors',
    'zcik':           '/content/loras/zcik.safetensors',
    'danziEngine':    '/content/loras/danzi_engine.safetensors',
    'edjit':          '/content/loras/edjit.safetensors',
    'dagasi':         '/content/loras/dagasi.safetensors',
}

DOWNLOADS = [
    ('https://civitai.com/api/download/models/2074658?fileId=1970963', MODEL_PATH,                    'Modelo SDXL'),
    ('https://civitai.com/api/download/models/1442564?fileId=1443706', LORA_REGISTRY['zonkpunch_base'], 'LoRA Zonkpunch Base'),
    ('https://civitai.red/api/download/models/2949119?fileId=2828573', LORA_REGISTRY['zonkpunch_1'],    'LoRA Zonkpunch Pack 1'),
    ('https://civitai.red/api/download/models/1817178?fileId=1717547', LORA_REGISTRY['zonkpunch_2'],    'LoRA Zonkpunch Pack 2'),
    ('https://civitai.red/api/download/models/2052381?fileId=1949320', LORA_REGISTRY['zonkpunch_3'],    'LoRA Zonkpunch Pack 3'),
    ('https://civitai.red/api/download/models/1883937?fileId=1783731', LORA_REGISTRY['zonkpunch_4'],    'LoRA Zonkpunch Pack 4'),
    ('https://civitai.red/api/download/models/1678987?fileId=1580798', LORA_REGISTRY['zonkpunch_5'],    'LoRA Zonkpunch Pack 5'),
    ('https://civitai.red/api/download/models/2576440?fileId=2463857', LORA_REGISTRY['zonkpunch_6'],    'LoRA Zonkpunch Pack 6'),
    ('https://civitai.com/api/download/models/1340337?fileId=1243946', LORA_REGISTRY['zcik'],           'LoRA Zcik'),
    ('https://civitai.com/api/download/models/3137729?fileId=3017792', LORA_REGISTRY['danziEngine'],    'LoRA Danzi Engine'),
    ('https://civitai.com/api/download/models/2729690?fileId=2615709', LORA_REGISTRY['edjit'],          'LoRA Edjit'),
    ('https://civitai.com/api/download/models/2307358?fileId=2198032', LORA_REGISTRY['dagasi'],         'LoRA Dagasi'),
]

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
if CIVITAI_API_KEY: headers['Authorization'] = f'Bearer {CIVITAI_API_KEY}'

for url, dest, label in DOWNLOADS:
    if os.path.exists(dest): continue
    print(f' ⬇ Descargando {label}...')
    with requests.get(url, headers=headers, stream=True, allow_redirects=True) as r:
        r.raise_for_status()
        with open(dest, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192): f.write(chunk)

FINAL_ZONKPUNCH_PATH = '/content/loras/zonkpunch_merged_final.safetensors'
if not os.path.exists(FINAL_ZONKPUNCH_PATH):
    print('🔮 FUSIONANDO EL PACK COMPLETO DE ZONKPUNCH EN CPU...')
    zp_keys = ['zonkpunch_base', 'zonkpunch_1', 'zonkpunch_2', 'zonkpunch_3', 'zonkpunch_4', 'zonkpunch_5', 'zonkpunch_6']
    merged_dict = {}
    
    # Usamos el Pack 4 como plantilla limpia de metadatos porque sabemos que diffusers lo lee de forma nativa
    template_dict = load_file(LORA_REGISTRY['zonkpunch_4'])
    
    # Mapeamos y adaptamos las capas de los otros paquetes conflictivos al formato limpio
    for k in template_dict.keys():
        tensors_list = []
        for zp_name in zp_keys:
            d = load_file(LORA_REGISTRY[zp_name])
            if k in d and d[k].shape == template_dict[k].shape:
                tensors_list.append(d[k].float())
            del d
        if tensors_list:
            merged_dict[k] = (sum(tensors_list) / len(tensors_list)).to(torch.float16)
        else:
            merged_dict[k] = template_dict[k].to(torch.float16)
            
    save_file(merged_dict, FINAL_ZONKPUNCH_PATH)
    del template_dict, merged_dict
    gc.collect()
    print('✅ ¡Pack unificado guardado con éxito sin errores de capas!')

# Actualizamos el registro apuntando a nuestro súper archivo unificado compatible
LORA_REGISTRY['zonkpunch'] = FINAL_ZONKPUNCH_PATH

from diffusers import StableDiffusionXLPipeline
print('\n🔧 Cargando modelo base SDXL en GPU...')
pipe = StableDiffusionXLPipeline.from_single_file(MODEL_PATH, torch_dtype=torch.float16, use_safetensors=True).to('cuda')
pipe.enable_attention_slicing()
try: pipe.enable_xformers_memory_efficient_attention()
except Exception: pass
print('\n✅ TODO LISTO — Ejecuta la Celda 2.')

## ▶ Celda 2 — Conexión Web e Inferencia con Protección de Timeout

In [ ]:
WEB_URL = 'https://art.kxd.es'
API_PORT = 7860

import io, base64, threading, time, re, subprocess, gc, os
import requests
from fastapi import FastAPI
import uvicorn, torch
from PIL import Image

app_api = FastAPI()
generation_lock = threading.Lock()

def run_inference(job_dict):
    with generation_lock:
        try: pipe.unload_lora_weights()
        except Exception: pass
        gc.collect()
        torch.cuda.empty_cache()

        lora_weights = job_dict.get('loraWeights') or {}
        active_loras, active_weights = [], []
        
        for name, weight in lora_weights.items():
            if name in LORA_REGISTRY and weight and weight > 0:
                adapter = f'lora_{name}'
                try:
                    pipe.load_lora_weights(
                        LORA_REGISTRY[name],
                        weight_name=os.path.basename(LORA_REGISTRY[name]),
                        adapter_name=adapter
                    )
                    active_loras.append(adapter)
                    active_weights.append(float(weight))
                except Exception as e:
                    print(f'  ⚠ LoRA {name}: {e}')
                    
        if active_loras:
            try: pipe.set_adapters(active_loras, adapter_weights=active_weights)
            except Exception as e: print(f'  ⚠ Error aplicando adaptadores: {e}')

        seed = job_dict.get('seed')
        if seed is None: seed = int(torch.randint(0, 2**32, (1,)).item())
        gen = torch.Generator('cuda').manual_seed(seed)

        start_time = time.time()
        def timeout_callback(pipe_obj, step, timestep, callback_kwargs):
            if (time.time() - start_time) > 50.0:
                raise TimeoutError("⚠️ Límite de 50 segundos superado.")
            return callback_kwargs

        try:
            with torch.autocast('cuda'):
                result = pipe(
                    prompt=job_dict.get('prompt', ''),
                    negative_prompt=job_dict.get('negativePrompt') or 'nsfw, blurry, low quality, deformed',
                    width=job_dict.get('width', 1024),
                    height=job_dict.get('height', 1024),
                    num_inference_steps=job_dict.get('steps', 30),
                    guidance_scale=job_dict.get('cfgScale', 7.0),
                    generator=gen,
                    callback_on_step_end=timeout_callback
                )
            img_out = result.images[0]
        except TimeoutError as te:
            print(f'\n❌ {te}')
            raise te
        finally:
            if active_loras:
                try: pipe.unload_lora_weights()
                except Exception: pass
            gc.collect()
            torch.cuda.empty_cache()

        return img_out, seed

threading.Thread(target=lambda: uvicorn.run(app_api, host='0.0.0.0', port=API_PORT, log_level='warning'), daemon=True).start()
time.sleep(2)

print('🌐 Creando túnel Cloudflare...')
cf_proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://localhost:{API_PORT}', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)

TUNNEL_URL = None
for _ in range(120):
    line = cf_proc.stdout.readline()
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if match: 
        TUNNEL_URL = match.group(0)
        break
if not TUNNEL_URL: raise RuntimeError('Error al abrir el túnel.')

SESSION_ID = None
try:
    resp = requests.post(f'{WEB_URL}/api/sessions/register', json={'tunnelUrl': TUNNEL_URL}, timeout=15)
    resp.raise_for_status()
    SESSION_ID = resp.json()['id']
    sync_ok = True
except Exception as e:
    sync_ok = False
    sync_error = str(e)

print('\n' + '━' * 52)
if sync_ok:
    print(f'✅ SINCRONIZADO — GPU activa en art.kxd.es\n Sesión #{SESSION_ID}')
else:
    print(f'⚠️ Error de conexión: {sync_error}\nEnlace manual:')
print(f' {TUNNEL_URL}\n' + '━' * 52)

def _afk():
    from IPython.display import display, Javascript
    while True:
        time.sleep(55)
        try: display(Javascript('document.dispatchEvent(new MouseEvent("mousemove",{bubbles:true}));'))
        except Exception: pass
threading.Thread(target=_afk, daemon=True).start()

HEADERS = {'Content-Type': 'application/json'}
start = time.time()
try:
    while time.time() - start < 12 * 3600:
        try:
            r = requests.get(f'{WEB_URL}/api/internal/next-job', params={'sessionId': SESSION_ID}, headers=HEADERS, timeout=10)
            r.raise_for_status()
            job = r.json().get('job')
        except Exception as e:
            time.sleep(3)
            continue
        if not job:
            time.sleep(3)
            continue

        jid = job['id']
        print(f'\n🎨 {jid[:8]}… | {job.get("width",1024)}×{job.get("height",1024)} | {job.get("steps",30)} pasos')
        try:
            t0 = time.time()
            img_obj, seed = run_inference(job)
            print(f' ✓ {time.time()-t0:.1f}s seed={seed}')
            buf = io.BytesIO()
            img_obj.save(buf, format='JPEG', quality=80)
            img_b64 = base64.b64encode(buf.getvalue()).decode()
            try:
                requests.post(f'{WEB_URL}/api/internal/job-complete', headers=HEADERS, timeout=60, json={'jobId': jid, 'sessionId': SESSION_ID, 'imageBase64': img_b64, 'seed': seed}).raise_for_status()
            except requests.exceptions.HTTPError as http_err:
                if http_err.response is not None and http_err.response.status_code == 413:
                    img_resized = img_obj.resize((512, 512), Image.Resampling.LANCZOS)
                    buf_small = io.BytesIO()
                    img_resized.save(buf_small, format='JPEG', quality=50)
                    img_b64_small = base64.b64encode(buf_small.getvalue()).decode()
                    requests.post(f'{WEB_URL}/api/internal/job-complete', headers=HEADERS, timeout=60, json={'jobId': jid, 'sessionId': SESSION_ID, 'imageBase64': img_b64_small, 'seed': seed}).raise_for_status()
                else: raise http_err
        except Exception as e:
            print(f' ❌ {e}')
            try: requests.post(f'{WEB_URL}/api/internal/job-failed', headers=HEADERS, timeout=10, json={'jobId': jid, 'sessionId': SESSION_ID, 'errorMessage': str(e)})
            except Exception: pass
except KeyboardInterrupt:
    print('\n🛑 Detenido.')
finally:
    try: requests.post(f'{WEB_URL}/api/internal/session-end', headers=HEADERS, timeout=5, json={'sessionId': SESSION_ID})
    except Exception: pass
    if cf_proc.poll() is None: cf_proc.terminate()